In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Sil2Gld_DimDim_FilenDocument V2"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Silver/Dim_Files" # ← Change source path
SOURCE_PATH = "abfss://Silver/Dim_Folders"
TARGET_PATH = "abfss://Gold/Dim_FilenDocument" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_gold_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_gold_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_gold_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 3, Finished, Available, Finished)

🔧 Initializing ntk_Sil2Gld_DimDim_FilenDocument V2...
🚀 Starting ntk_Sil2Gld_DimDim_FilenDocument V2


### Paths of file reading and saving 

In [2]:
# Section --- Importing modules and Defining Variable for standard usage source / target.
from pyspark.sql import SparkSession
from datetime import datetime
import os

# Base source path (up to Files level)
base_source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"
# Base target path  
base_target_path  = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"

# Get today's date and format it
today = datetime.now()
today = datetime.now()  
from datetime import datetime, timedelta
###today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")  

# Build dynamic source folder structure
sourcefolder_structure = f"/PreGold_Reporting"    
# Complete source path
complete_source_path = f"{base_source_path}/{sourcefolder_structure}"
# Filename
source_filename = "Dim_Files.parquet"
# FolderName
source_foldername = "Dim_Folders.parquet"

# Build dynamic target folder structure
targetfolder_structure = f"/PreGold_Reporting"   
# Complete target path
complete_target_path = f"{base_target_path}/{targetfolder_structure}"
# TargetFilename
target_filename = "Dim_FilenDocument.parquet"
print(f"Variables created and session started.")


StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 4, Finished, Available, Finished)

Variables created and session started.


#### Source and Target path

In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("SilverToGold_FileReader").getOrCreate()

# Full Source  file path
full_source_filepath = f"{complete_source_path}/{source_filename}"
full_source_folderpath = f"{complete_source_path}/{source_foldername}"

# Full Target  file path
full_target_filepath = f"{complete_target_path}/{target_filename}"

print(f"Source: {full_source_filepath} & {full_source_folderpath}")
print(f"Target: {full_target_filepath}")

StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 5, Finished, Available, Finished)

Source: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Files.parquet & abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Folders.parquet
Target: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_FilenDocument.parquet


In [4]:
# mssparkutils.notebook.run("nbk_dimuser_validations", 60)

StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 6, Finished, Available, Finished)

### Source file Reading

In [5]:

# ===== READ PROCESS =====
try:
    # Read the Parquet file from Silver layer
    print("Reading from Gold layer: File...")
    df_goldfiles = spark.read.parquet(full_source_filepath)
    print(f"Input file read with total records: {df_goldfiles.count()}")
    #df_goldfiles.show(1)
    df_goldfiles.printSchema()

    print("Reading from Gold layer: Folder...")
    df_goldfolders = spark.read.parquet(full_source_folderpath)
    print(f"Input file read with total records: {df_goldfolders.count()}")
    #df_goldfolders.show(1)
    df_goldfolders.printSchema()

    # STAGE 1: PRE-VALIDATION GATE - CRITICAL CHECKPOINT
    # validation_passed, validation_report = step1_validate_source_quality(df_silver)

    # STOP PIPELINE IF CRITICAL VALIDATION FAILURES
    # if not validation_passed:
    #     print("\n🛑 ETL PIPELINE STOPPED - Critical validation failures detected")
    #     print("Fix source data issues before proceeding")
    #     raise Exception("Pre-validation failed - ETL pipeline terminated")
        
    # print("\n🚀 Pre-validation passed - Proceeding with transformation...")

except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Content Reading  completed!")


StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 7, Finished, Available, Finished)

Reading from Gold layer: File...
Input file read with total records: 2162
root
 |-- FileKey: string (nullable = true)
 |-- LibraryKey: string (nullable = true)
 |-- SiteKey: string (nullable = true)
 |-- ObjectID: string (nullable = true)
 |-- ItemID: integer (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- FileName: string (nullable = true)
 |-- FilePath: string (nullable = true)
 |-- FileExtension: string (nullable = true)
 |-- FileType: string (nullable = true)
 |-- ContentType: string (nullable = true)
 |-- Owner: string (nullable = true)
 |-- DCLocation: string (nullable = true)
 |-- GeoLocation: string (nullable = true)
 |-- DataRedundancy: string (nullable = true)
 |-- Version: double (nullable = true)
 |-- IsCurrentVersion: boolean (nullable = true)
 |-- SensitivityLabel: string (nullable = true)
 |-- Classification: string (nullable = true)
 |-- ComplianceTag: string (nullable = true)
 |-- RetentionPolicy: string (nullable = true)
 |-- IsActive: string (nullable = 

In [6]:
# df_goldfiles_transformed.printSchema()
# df_goldfolders_transformed.printSchema()

StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 8, Finished, Available, Finished)

In [7]:
# # Cache for performance
# target_df.cache()
# print(target_df.count())

StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 9, Finished, Available, Finished)

root
 |-- FileKey: string (nullable = true)
 |-- LibraryKey: string (nullable = true)
 |-- SiteKey: string (nullable = true)
 |-- ObjectID: string (nullable = true)
 |-- ItemID: integer (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- FileName: string (nullable = true)
 |-- FilePath: string (nullable = true)
 |-- FileExtension: string (nullable = true)
 |-- FileType: string (nullable = true)
 |-- ContentType: string (nullable = true)
 |-- Owner: string (nullable = true)
 |-- DCLocation: string (nullable = true)
 |-- GeoLocation: string (nullable = true)
 |-- DataRedundancy: string (nullable = true)
 |-- Version: double (nullable = true)
 |-- IsCurrentVersion: boolean (nullable = true)
 |-- SensitivityLabel: string (nullable = true)
 |-- Classification: string (nullable = true)
 |-- ComplianceTag: string (nullable = true)
 |-- RetentionPolicy: string (nullable = true)
 |-- IsActive: string (nullable = true)
 |-- ApprovalStatus: string (nullable = true)
 |-- IsSharedExternall

In [8]:
# ===== COMBINE df_goldfiles AND df_goldfolders INTO SINGLE DATAFRAME =====
# Following updated mapping instructions and data type requirements

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when
from pyspark.sql.types import StringType, DateType, BooleanType

print("🔄 Starting DataFrame combination process...")

# ===== STEP 1: TRANSFORM df_goldfiles =====
print("📋 Step 1: Transforming df_goldfiles...")

df_goldfiles_transformed = df_goldfiles \
    .withColumn("ObjectKey", col("FileKey").cast(StringType())) \
    .withColumn("LibraryKey", col("LibraryKey").cast(StringType())) \
    .withColumn("SiteKey", col("SiteKey").cast(StringType())) \
    .withColumn("ObjectID", col("ObjectID").cast(StringType())) \
    .withColumn("DocumentLibraryID", col("ContentType")) \
    .withColumn("SiteID", col("SiteID").cast(StringType())) \
    .withColumn("Version", col("Version").cast(StringType())) \
    .withColumn("CreatedDate", col("CreatedDate").cast(DateType())) \
    .withColumn("ModifiedDate", col("ModifiedDate").cast(DateType())) \
    .withColumn("FolderName", col("FileName")) \
    .withColumn("FolderPath", col("FilePath")) \
    .withColumn("ParentFolderName", lit("").cast(StringType())) \
    .withColumn("HasSubFolders", lit("").cast(StringType())) \
    .select(
        "ObjectKey",
        "LibraryKey",
        "SiteKey",
        "ObjectID",
        "ItemID", 
        "SiteID",
        "DocumentLibraryID",
        "FolderName",           # Mapped from FileName
        "ParentFolderName",     # Set blank
        "HasSubFolders",        # Set blank  
        "FolderPath",           # Mapped from FilePath
        "FileExtension",
        "FileType",
        "ContentType",
        "Owner",
        "DCLocation",
        "GeoLocation", 
        "DataRedundancy",
        "Version",
        "IsCurrentVersion",
        "SensitivityLabel",
        "Classification",
        "ComplianceTag",
        "RetentionPolicy",
        "IsActive",
        "ApprovalStatus",
        "IsSharedExternally",
        "CreatedBy",
        "CreatedDate", 
        "ModifiedBy",
        "ModifiedDate",
        "SnapshotDate"
    )

print(f"✅ df_goldfiles transformed: {df_goldfiles_transformed.count()} records")

# ===== STEP 2: TRANSFORM df_goldfolders =====
print("📋 Step 2: Transforming df_goldfolders...")

df_goldfolders_transformed = df_goldfolders \
    .withColumn("ObjectKey", col("FolderKey").cast(StringType())) \
    .withColumn("LibraryKey", col("LibraryKey").cast(StringType())) \
    .withColumn("SiteKey", col("SiteKey").cast(StringType())) \
    .withColumn("ObjectID", col("ObjectID").cast(StringType())) \
    .withColumn("SiteID", col("SiteID").cast(StringType())) \
    .withColumn("CreatedBy", col("CreatedBy").cast(StringType())) \
    .withColumn("ModifiedBy", col("ModifiedBy").cast(StringType())) \
    .withColumn("FileExtension", lit("").cast(StringType())) \
    .withColumn("FileType", lit("Folder").cast(StringType())) \
    .withColumn("IsCurrentVersion", lit(True).cast(BooleanType())) \
    .withColumn("ApprovalStatus", lit("Published").cast(StringType())) \
    .select(
        "ObjectKey",
        "LibraryKey",
        "SiteKey",
        "ObjectID",
        "ItemID",
        "SiteID", 
        "DocumentLibraryID",
        "FolderName",
        "ParentFolderName",
        "HasSubFolders",
        "FolderPath",
        "FileExtension",        # Set blank
        "FileType",             # Set "Folder"
        "ContentType",
        "Owner",
        "DCLocation",
        "GeoLocation",
        "DataRedundancy",
        "Version",
        "IsCurrentVersion",     # Set True
        "SensitivityLabel",
        "Classification", 
        "ComplianceTag",
        "RetentionPolicy",
        "IsActive",
        "ApprovalStatus",       # Set "Published"
        "IsSharedExternally",
        "CreatedBy",
        "CreatedDate",
        "ModifiedBy", 
        "ModifiedDate",
        "SnapshotDate"
    )

print(f"✅ df_goldfolders transformed: {df_goldfolders_transformed.count()} records")

# ===== STEP 3: UNION BOTH DATAFRAMES =====
print("🔗 Step 3: Combining dataframes...")

df_combined = df_goldfiles_transformed.union(df_goldfolders_transformed)

print(f"🎉 DataFrames combined successfully!")
print(f"📊 Total records in combined dataframe: {df_combined.count()}")

# ===== STEP 4: VERIFY SCHEMA =====
print("\n📋 Step 4: Verifying combined schema...")
print("Combined DataFrame Schema:")
df_combined.printSchema()



StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 10, Finished, Available, Finished)

🔄 Starting DataFrame combination process...
📋 Step 1: Transforming df_goldfiles...
✅ df_goldfiles transformed: 2162 records
📋 Step 2: Transforming df_goldfolders...
✅ df_goldfolders transformed: 313 records
🔗 Step 3: Combining dataframes...
🎉 DataFrames combined successfully!
📊 Total records in combined dataframe: 2475

📋 Step 4: Verifying combined schema...
Combined DataFrame Schema:
root
 |-- ObjectKey: string (nullable = true)
 |-- LibraryKey: string (nullable = true)
 |-- SiteKey: string (nullable = true)
 |-- ObjectID: string (nullable = true)
 |-- ItemID: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- DocumentLibraryID: string (nullable = true)
 |-- FolderName: string (nullable = true)
 |-- ParentFolderName: string (nullable = true)
 |-- HasSubFolders: string (nullable = true)
 |-- FolderPath: string (nullable = true)
 |-- FileExtension: string (nullable = true)
 |-- FileType: string (nullable = true)
 |-- ContentType: string (nullable = true)
 |-- Owner: str

In [9]:
# ===== STEP 5: SAMPLE DATA VERIFICATION =====
print("\n🔍 Step 5: Sample data verification...")

print("Sample Files data (FileType != 'Folder'):")
df_combined.filter(col("FileType") != "Folder").select(
    "ObjectID", "FolderName", "FileType", "FileExtension", "ApprovalStatus", "IsCurrentVersion"
).show(3, truncate=False)

print("Sample Folders data (FileType = 'Folder'):")
df_combined.filter(col("FileType") == "Folder").select(
    "ObjectID", "FolderName", "FileType", "ParentFolderName", "HasSubFolders", "ApprovalStatus"
).show(3, truncate=False)

# ===== STEP 6: DATA QUALITY CHECKS =====
print("\n✅ Step 6: Data quality summary...")

files_count = df_combined.filter(col("FileType") != "Folder").count()
folders_count = df_combined.filter(col("FileType") == "Folder").count()
total_count = df_combined.count()

print(f"📁 Files: {files_count}")
print(f"📂 Folders: {folders_count}")  
print(f"📊 Total: {total_count}")
print(f"🔍 Verification: {files_count + folders_count == total_count}")

# Check for nulls in key fields
print("\nNull value check in key fields:")
key_fields = ["ObjectID", "ItemID", "SiteID", "FolderName"]
for field in key_fields:
    null_count = df_combined.filter(col(field).isNull()).count()
    print(f"  {field}: {null_count} nulls")

print("\n🚀 DataFrame combination completed successfully!")
print("Combined dataframe available as: df_combined")

# ===== OPTIONAL: SHOW FINAL COLUMN LIST =====
print(f"\n📋 Final DataFrame Columns ({len(df_combined.columns)}):")
for i, col_name in enumerate(df_combined.columns, 1):
    print(f"  {i:2}. {col_name}")

StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 11, Finished, Available, Finished)


🔍 Step 5: Sample data verification...
Sample Files data (FileType != 'Folder'):
+------------------------------------+-----------------------------+-----------+-------------+--------------+----------------+
|ObjectID                            |FolderName                   |FileType   |FileExtension|ApprovalStatus|IsCurrentVersion|
+------------------------------------+-----------------------------+-----------+-------------+--------------+----------------+
|00ad7bde-ff7a-423a-b3b6-8c718e93ab15|Box_Full.png.deploy          |Other      |deploy       |Published     |true            |
|0b9525e7-2d0c-469a-ac82-09dc505abb6c|MSRA_MSTeamsEss_Seg_0106.xlsx|Spreadsheet|xlsx         |Published     |true            |
|25373365-0440-46a7-9e97-b557fbfdc491|Windata1214.xlsx             |Spreadsheet|xlsx         |Published     |true            |
+------------------------------------+-----------------------------+-----------+-------------+--------------+----------------+
only showing top 3 rows

Sampl

In [10]:
# ===== WRITE PROCESS =====

df_combined

try:
    # Write to Gold layer (PreGold_Reporting)
    print(f"Writing to Gold layer: {full_target_filepath}")
    
    df_combined.write \
        .mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(full_target_filepath)
    
    print(f"Successfully written to: {full_target_filepath}")
    
    # Verify the written file
    df_verify = spark.read.parquet(full_target_filepath)
    print(f"Verification - Target record count: {df_verify.count()}")
    
except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_filepath}")

df_combined.printSchema()
print("Process completed!")


StatementMeta(, a3e867f8-373c-44f5-a741-56de4e34b6d6, 12, Finished, Available, Finished)

Writing to Gold layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_FilenDocument.parquet
Successfully written to: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_FilenDocument.parquet
Verification - Target record count: 2475
root
 |-- ObjectKey: string (nullable = true)
 |-- LibraryKey: string (nullable = true)
 |-- SiteKey: string (nullable = true)
 |-- ObjectID: string (nullable = true)
 |-- ItemID: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- DocumentLibraryID: string (nullable = true)
 |-- FolderName: string (nullable = true)
 |-- ParentFolderName: string (nullable = true)
 |-- HasSubFolders: string (nullable = true)
 |-- FolderPath: string (nullable = true)
 |-- FileExtension: string (nullable = true)
 |-- FileType: string (nullable = true)
 |-- ContentType: string (nullable = true)
 |-- Owner: string (nullable = true)
 |-- 

In [13]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = df_goldfiles.count()
    rows_read = df_goldfolders.count()
    rows_written = df_combined.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, a4f1e381-88b8-41bb-b0fb-a1a843122985, 15, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_Sil2Gld_DimDim_FilenDocument V2...
✅ ntk_Sil2Gld_DimDim_FilenDocument V2 completed successfully (208s)
🎉 ntk_Sil2Gld_DimDim_FilenDocument V2 pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 1 → 2,163
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_Sil2Gld_DimDim_FilenDocument V2:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_Sil2Gld_DimDim_FilenDocument V2 logging completed!
